# Quickstart Demo: Data Center Water Scarcity Assessment

### Minimal Working Example for Peer Review & Replication
**Estimated Execution Time:** < 5 seconds | **Dependencies:** Standard scientific Python (`pandas`, `numpy`, `matplotlib`)

This notebook provides a self-contained demonstration of the methodology developed in this study, including **energy and water use analysis (Notebook 2)** with **water scarcity assessment (Notebook 3)**.

It uses a simulated dataset of five data centers of which some are located in close proximity, as well as with simulated hydrological basin data.


## 1. Setup and Imports

Imports the analytical functions from the project's [`functions.water_scarcity`](functions/water_scarcity) module.


In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import water scarcity functions
from functions.water_scarcity.analysis import (
    summing_dc_water_use_per_extraction_site,
    repeat_rows_for_months,
    compute_efr,
    calculate_water_scarcity,
    get_water_scarcity_summary,
    MONTH_NAMES,
)

# Output directory for demo figures
DEMO_OUTPUT_DIR = Path("data/outputs/demo/")

## 2. Load Simulated Facilities & Hydrological Data

The simulated dataset comprises:
1. **Facilities** (`simulated_facilities.csv`):
   - **Desert Hyperscale Campus A** (80 MW, evaporative cooling towers, arid climate ASHRAE 2B).
   - **Desert Colocation Facility B** (40 MW planned, evaporative cooling towers, co-located with Campus A).
   - **Midwest Enterprise Hub C** (50 MW, air-cooled chillers + economizer, temperate climate ASHRAE 4A).
   - **Midwest Edge Center D** (8 MW, air-cooled chillers).
   - **Desert CCGT Power Station** (450 MW natural gas power plant serving the regional grid).
2. **Hydrological Scenarios** (`simulated_hydrology.csv`):
   - **Arid River Basin (Southwest)**: Spring runoff peak, severely constrained summer discharge, high agricultural baseline abstractions.
   - **Perennial River Basin (Midwest)**: Abundant discharge throughout all 12 months.
   - **Climate Scenarios**: `hist` (Historical baseline) and `2_0C` (+2.0°C warming).


In [27]:
# Load simulated datasets
facilities_path = Path("data/inputs/demo/simulated_facilities.csv")
hydrology_path = Path("data/inputs/demo/simulated_hydrology.csv")

facilities = pd.read_csv(facilities_path)
hydrology = pd.read_csv(hydrology_path)

print(f"Loaded {len(facilities)} simulated facilities:")
facilities[["id", "name", "type", "critical_power_mw", "ashrae_zone", "cooling_tech", "river_basin_name"]]


Loaded 5 simulated facilities:


,id,name,type,critical_power_mw,ashrae_zone,cooling_tech,river_basin_name
0,DC_Desert_Hyperscale_1,Desert Hyperscale Campus A,data_center,80.0,2B,Evaporative Cooling Tower,Arid River Basin (Southwest)
1,DC_Desert_Colo_2,Desert Colocation Facility B,data_center,40.0,2B,Evaporative Cooling Tower,Arid River Basin (Southwest)
2,DC_Temperate_Cloud_3,Midwest Enterprise Hub C,data_center,50.0,4A,Air-Cooled Chiller + Economizer,Perennial River Basin (Midwest)
3,DC_Temperate_Edge_4,Midwest Edge Center D,data_center,8.0,4A,Air-Cooled Chiller,Perennial River Basin (Midwest)
4,PP_CombinedCycleGas_1,Desert CCGT Power Station,power_plant,450.0,2B,Closed-loop Cooling Tower,Arid River Basin (Southwest)


## 3. Direct & Indirect Water Footprint Estimation

For each data center, annual electricity demand and water footprints are estimated following the methodology from Notebook 2:

$$\text{Annual Electricity Use (MWh)} = \text{IT Power (MW)} \times \text{PUE} \times \text{Load Factor} \times 8760\ \text{hours/year}$$

$$\text{Direct Water Use } (\text{m}^3) = \text{IT Power (MW)} \times \text{WUE}_{\text{withdrawal}}\ (\text{m}^3/\text{MWh}) \times \text{Load Factor} \times 8760\ \text{hours/year}$$

$$\text{Indirect Water Use } (\text{m}^3) = \text{Annual Electricity Use (MWh)} \times \text{Grid Water Intensity}\ (\text{m}^3/\text{MWh})$$

$$\text{Total Water Footprint } (\text{m}^3) = \text{Direct Water Use } (\text{m}^3) + \text{Indirect Water Use } (\text{m}^3)$$


In [28]:
# Calculate energy and water footprints
facilities["annual_electricity_use_MWh"] = (
    facilities["critical_power_mw"] * facilities["PUE"] * facilities["load_factor"] * 8760
)
facilities["annual_direct_water_use_m3"] = (
    facilities["critical_power_mw"] * facilities["WUE_withdrawal"] * facilities["load_factor"] * 8760
)
facilities["annual_direct_water_consumption_m3"] = (
    facilities["critical_power_mw"] * facilities["WUE_consumption"] * facilities["load_factor"] * 8760
)
facilities["indirect_water_use_m3"] = (
    facilities["annual_electricity_use_MWh"] * facilities["grid_water_intensity_m3_per_MWh"]
)
facilities["total_water_use_m3"] = (
    facilities["annual_direct_water_use_m3"] + facilities["indirect_water_use_m3"]
)

# Set the primary water use metric for spatial assignment
facilities["water_use_m3"] = facilities["total_water_use_m3"]

# Display footprint breakdown
facilities[[
    "id", "type", "critical_power_mw",
    "annual_electricity_use_MWh", "annual_direct_water_use_m3",
    "indirect_water_use_m3", "total_water_use_m3"
]].style.format({
    "critical_power_mw": "{:.1f} MW",
    "annual_electricity_use_MWh": "{:,.0f} MWh",
    "annual_direct_water_use_m3": "{:,.0f} m³",
    "indirect_water_use_m3": "{:,.0f} m³",
    "total_water_use_m3": "{:,.0f} m³",
})


,id,type,critical_power_mw,annual_electricity_use_MWh,annual_direct_water_use_m3,indirect_water_use_m3,total_water_use_m3
0,DC_Desert_Hyperscale_1,data_center,80.0 MW,"657,000 MWh","946,080 m³","1,281,150 m³","2,227,230 m³"
1,DC_Desert_Colo_2,data_center,40.0 MW,"313,958 MWh","453,768 m³","612,219 m³","1,065,987 m³"
2,DC_Temperate_Cloud_3,data_center,50.0 MW,"387,630 MWh","114,975 m³","542,682 m³","657,657 m³"
3,DC_Temperate_Edge_4,data_center,8.0 MW,"55,573 MWh","18,221 m³","77,803 m³","96,024 m³"
4,PP_CombinedCycleGas_1,power_plant,450.0 MW,"2,365,200 MWh",0 m³,"4,612,140 m³","4,612,140 m³"


## 4. Aggregating to Extraction Sites

Individual data centers may withdraw water volumes that appear modest when evaluated in isolation. However, data centers frequently cluster geographically around fiber optic hubs and power junctions, drawing from the same hydrological sub-basin or river reaches.

Using `summing_dc_water_use_per_extraction_site(...)`:
1. Facilities within the same hydrological extraction buffer are aggregated.
2. The `shared_extraction_site` flag identifies co-located facilities.
3. Cumulative demand is divided into average monthly withdrawal (`dc_cumulative_monthly_water_use_m3`).


In [29]:
# Filter data centers for hydrological cluster assessment
dcs = facilities[facilities["type"] == "data_center"].copy()

# Aggregate cumulative data center water demand per extraction point
clustered_dcs = summing_dc_water_use_per_extraction_site(dcs)

clustered_dcs[[
    "id", "river_basin_name", "lat_water_extraction", "lon_water_extraction",
    "shared_extraction_site", "dc_annual_water_use_m3", "dc_cumulative_monthly_water_use_m3"
]].style.format({
    "dc_annual_water_use_m3": "{:,.0f} m³/yr",
    "dc_cumulative_monthly_water_use_m3": "{:,.0f} m³/mo",
})


,id,river_basin_name,lat_water_extraction,lon_water_extraction,shared_extraction_site,dc_annual_water_use_m3,dc_cumulative_monthly_water_use_m3
0,DC_Desert_Hyperscale_1,Arid River Basin (Southwest),33.400000,-112.100000,True,"2,227,230 m³/yr","274,435 m³/mo"
1,DC_Desert_Colo_2,Arid River Basin (Southwest),33.400000,-112.100000,True,"1,065,987 m³/yr","274,435 m³/mo"
2,DC_Temperate_Cloud_3,Perennial River Basin (Midwest),41.850000,-87.650000,True,"657,657 m³/yr","62,807 m³/mo"
3,DC_Temperate_Edge_4,Perennial River Basin (Midwest),41.850000,-87.650000,True,"96,024 m³/yr","62,807 m³/mo"


## 5. Monthly Hydrological Coupling & Environmental Flow Requirements (EFR)

To evaluate seasonal water stress:
1. Rows are expanded for all 12 calendar months using `repeat_rows_for_months(...)`.
2. Monthly river discharge ($Q$) and background human abstraction ($A$) are attached.
3. **Environmental Flow Requirements (EFR)** are computed using `compute_efr(...)` via the **Variable Monthly Flow (VMF)** method (Pastor et al. 2014):
   - **Low-flow months** ($\text{Monthly Flow} / \text{Annual Flow} \le 0.4$): $\text{EFR} = 60\%$ of flow
   - **High-flow months** ($\text{Monthly Flow} / \text{Annual Flow} > 0.8$): $\text{EFR} = 30\%$ of flow
   - **Intermediate-flow months**: $\text{EFR} = 45\%$ of flow


In [30]:
# Expand to 12 monthly time steps
monthly_dcs = repeat_rows_for_months(clustered_dcs)

# Merge with historical hydrological data
hydro_hist = hydrology[hydrology["scenario"] == "hist"]
merged_hist = monthly_dcs.merge(
    hydro_hist[["lat_water_extraction", "lon_water_extraction", "month", "discharge_m3_median", "abstraction_m3_median"]],
    on=["lat_water_extraction", "lon_water_extraction", "month"],
    how="left",
)

# Compute Environmental Flow Requirements (EFR)
efr_hist = compute_efr(merged_hist)

# Display sample of computed EFR for the arid basin
sample_arid_efr = efr_hist[efr_hist["id"] == "DC_Desert_Hyperscale_1"][
    ["month", "discharge_m3_median", "abstraction_m3_median", "mean_monthly_flow", "season", "EFR_ratio", "EFR_m3_month"]
]
sample_arid_efr.style.format({
    "discharge_m3_median": "{:,.0f}",
    "abstraction_m3_median": "{:,.0f}",
    "mean_monthly_flow": "{:,.0f}",
    "EFR_ratio": "{:.2f}",
    "EFR_m3_month": "{:,.0f}",
})


,month,discharge_m3_median,abstraction_m3_median,mean_monthly_flow,season,EFR_ratio,EFR_m3_month
0,Jan,"1,800,000","120,000","1,920,000",high,0.30,"576,000"
1,Feb,"2,200,000","150,000","2,350,000",high,0.30,"705,000"
2,Mar,"2,800,000","220,000","3,020,000",high,0.30,"906,000"
3,Apr,"1,500,000","350,000","1,850,000",high,0.30,"555,000"
4,May,"800,000","450,000","1,250,000",high,0.30,"375,000"
5,Jun,"350,000","400,000","750,000",intermediate,0.45,"337,500"
6,Jul,"220,000","360,000","580,000",intermediate,0.45,"261,000"
7,Aug,"180,000","300,000","480,000",low,0.60,"288,000"
8,Sep,"250,000","220,000","470,000",low,0.60,"282,000"
9,Oct,"500,000","160,000","660,000",intermediate,0.45,"297,000"


## 6. Water Scarcity & Vulnerability Assessment

Water scarcity is evaluated using the **Water Scarcity Index (WSI)**:

$$\text{WSI}_{\text{baseline}} = \frac{\text{Abstraction}}{(\text{Discharge} + \text{Abstraction}) - \text{EFR}}$$

$$\text{WSI}_{\text{with DC}} = \frac{\text{Abstraction} + \text{DC Cumulative Monthly Water Use}}{(\text{Discharge} + \text{Abstraction}) - \text{EFR}}$$

A month is defined as **water scarce** when $\text{WSI} > 1.0$ 

We now evaluate both the **Historical baseline** and the **+2.0°C Global Warming Scenario**:


In [32]:
columns_for_summary = [
    "id", "latitude", "longitude", "critical_power_mw", "operational",
    "power_grid_zone", "dc_cumulative_monthly_water_use_m3",
    "shared_extraction_site", "type", "lon_water_extraction", "lat_water_extraction",
]

scenario_results = {}
for scen in ["hist", "2_0C"]:
    hydro_scen = hydrology[hydrology["scenario"] == scen]
    merged_scen = monthly_dcs.merge(
        hydro_scen[["lat_water_extraction", "lon_water_extraction", "month", "discharge_m3_median", "abstraction_m3_median"]],
        on=["lat_water_extraction", "lon_water_extraction", "month"],
        how="left",
    )
    
    # 1. Compute EFR
    efr_scen = compute_efr(merged_scen)
    
    # 2. Compute Water Scarcity Index
    scarcity_scen = calculate_water_scarcity(
        efr_scen,
        abstraction_column="abstraction_m3_median",
        discharge_column="discharge_m3_median",
        efr_column="EFR_m3_month",
        efr_sensitivity_analysis=True,
    )
    
    # 3. Summarize by location
    summary_scen = get_water_scarcity_summary(scarcity_scen, columns_for_summary, efr_sensitivity_analysis=True)
    summary_scen["scenario"] = scen
    scenario_results[scen] = (scarcity_scen, summary_scen)

# Combine scenario summaries
summary_all = pd.concat([scenario_results["hist"][1], scenario_results["2_0C"][1]], ignore_index=True)

summary_all[[
    "scenario", "id", "critical_power_mw", "shared_extraction_site",
    "months_WSI", "months_WSI_dc", "dc_direct_increases_WSI_months", "dc_direct_increase_months_count"
]]


# Export the simulated results
summary_all.to_csv(f"{DEMO_OUTPUT_DIR}/demo_results_summary")

### Example Insights from Scarcity Results:
1. **Exacerbating scarcity**: In the Arid River Basin, baseline background water scarcity occurs during 3 summer months under historical conditions. When the cumulative water demand of the data center cluster is added, **water scarcity expands to 5 months** (`dc_direct_increases_WSI_months = True`).
2. **Climate Compounding**: Under the +2.0°C warming scenario, reduced summer runoff increases baseline scarcity to 4 months, and data center demand pushes the basin into **6 months of severe water scarcity**.
3. **Importance of site location**: The Perennial Midwest basin maintains $\text{WSI} < 1.0$ across all 12 months in both historical and warming scenarios, demonstrating that siting is a key determinant for whether data center water use is an ecological concern.
